# STUDY NOTEBOOK 5: Layer 5, The Tool

**Who this is for:** a team member who needs to know Layer 5: the part officers and field staff actually touch. You must be able to explain it to the guide, answer questions, and read `05_tool.ipynb` and its helper files without fear.

**What you need before you start:** basic Python, and "what is an average" and "what is a probability". Everything else (the normal curve, shrinkage, hashes) is taught here from zero.

**How to study this notebook**

1. Read top to bottom once and run every code cell (Shift+Enter). The cells use small toy numbers so you can see each idea.
2. After each section, cover the page and say the **Say it to the guide** box in your own words.
3. At the end: the **numbers to remember** table, the **glossary** and the **question list**. Mug those.
4. Then open the real `05_tool.ipynb` and follow it. Section numbers here match its sections 1 to 10.

> **bold** = a term you must know. `code font` = a name that exists in the real code (files `intake.py`, `card.py`, `views.py` in `notebooks/paddybird/`).

## 0. Where Layer 5 sits, and what it does NOT do

**The one-line summary:** Layer 5 takes the answers produced by Layers 1 to 4 and shows them to a human in a way they can act on.

| Layer | Produces |
|---|---|
| 1 Data and platform | satellite chips, units, roads, receipts |
| 2 Mapping | probability that each pixel is flooded cropland |
| 3 Estimation | for each unit: a flooded share **theta_u**, an uncertainty **s_u**, a **call**, and **P_u** |
| 4 Field planning | which villages to check, and the routes |
| **5 Tool** | **shows all of that: unit-calls table, map, evidence card, route view, offline entry, farmer report check** |

**Layer 5 fits no model.** It does not learn anything. So there is **no train/test split in Layer 5**, and the only "metric" is structural: does every unit have a call, and does every receipt re-derive? Say this early, because examiners ask "what is the accuracy of Layer 5?" and the answer is "Layer 5 has no accuracy, it displays Layers 2 and 3. Its check is integrity."

**The big rule (again): no abstain state.** The earlier prototype could say "not enough evidence". The new tool **never** does. Every unit gets a call, either **above** or **below** the loss line, and a number **P_u** says how far to trust it. There is not even an "abstain" word anywhere in a card; a check in the notebook asserts that.

**The loss line** is **tau = 0.25**: 25 percent of a unit's paddy area flooded. (The project's setting, modelled on the localised-loss rule in the crop-insurance scheme PMFBY.)

## 1. The formulas behind the numbers (you MUST understand these, the tool is built on them)

Layer 5 shows four numbers for each unit. You do not need to derive Layer 3, but you must know what these mean and how they connect. We build them step by step using the simple strata formula, then the clarification box explains how it relates to Layer 3's real estimator.

> **Important clarification (so nobody catches you out).** Layer 3's headline estimator is a more advanced one, called prediction-powered inference, taught in Study notebook 3. The simple formulas in this section, `p_uh = (k + m pi_h) / (n + m)` and the variance below, are the **older partial-report formula**. Layer 3 keeps it as a comparison baseline named `report_formula`. The tool uses it for live updates only when a unit carries the strata columns `W0, W1, k0, n0, k1, n1` (this is how the 12-unit fixture works). For the **34 real units** in our run, the tool reads theta and s **straight from Layer 3's `unit_calls.csv`**, which has no strata columns, so new checks update those units with the **Beta update** (section 7 of this notebook), not with the strata formula. What is common to everything: **call = theta above tau, and P_u = Phi((theta - tau) / s)**. Only how theta and s are produced differs.

### 1.1 Strata and the unit's flooded share (theta_u)

For one unit u, we split its paddy area into **strata** (groups). Here there are **two map strata**:

* stratum **0 = "map dry"**: places the radar map says are not flooded;
* stratum **1 = "map flooded"**: places the map says are flooded.

* **W_uh** = the **share of the unit's paddy area** that lies in stratum h. (W0 + W1 = 1.) The map gives us this.
* **p_uh** = the **fraction of places in stratum h that are truly flooded**. The map is not perfect, so even "map dry" places are sometimes flooded and "map flooded" places are sometimes dry. We estimate p_uh from the field checks.

Then the unit's flooded share is a **weighted average**:

**theta_u = W_u0 * p_u0 + W_u1 * p_u1**   (written in the notebook as `theta_u = sum_h W_uh p_uh`)

Plain English: "share of area in each group, times how flooded that group really is, added up."

### 1.2 Shrinkage: how to estimate p_uh from a few checks

Suppose in a stratum we checked 2 places and both were flooded. Is p = 100 percent? That is silly from 2 checks. So we **shrink** the estimate toward a sensible **prior** (a belief we had before looking at this unit).

**p_uh = (k_uh + m * pi_h) / (n_uh + m)**

* **k_uh** = number (weighted) of checks that were flooded; **n_uh** = number (weighted) of checks done in the stratum.
* **pi_h** = the **prior**: how often a place in that stratum is flooded in general, learned in Layer 2 calibration. Values used: **pi_0 = 0.08** (map dry: flooded 8 percent of the time), **pi_1 = 0.75** (map flooded: flooded 75 percent of the time).
* **m** = the **shrinkage strength = 10**. Think of it as "the prior is worth 10 imaginary checks".

Careful: this **pi_h is a different "pi" from the inclusion probability pi** of Layer 1. Same letter, different meaning. In the code the prior is `PRIOR_PI`, the inclusion probability is the column `pi`. Say which one you mean.

**Intuition:** with no checks (n=0) you get the prior. With very many checks you get the observed rate. In between you get a blend. That is exactly what "shrink" means.

### 1.3 The uncertainty s_u

**s_u** says how uncertain theta_u is. In the code:

`variance = sum_h  W_uh^2 * p_uh * (1 - p_uh) / (n_uh + m + 1)`, and `s_u = sqrt(variance)`.

You do not need to derive it. Know the behaviour: **more checks means smaller s_u**; a stratum with a middling p (near 0.5) is more uncertain than one near 0 or 1.

### 1.4 The call and P_u

* **The call:** `call_u = "above" if theta_u > tau else "below"`. Always one of the two.
* **P_u** = the probability that the unit's **true** flooded share is above tau:

**P_u = Phi((theta_u - tau) / s_u)**

**Phi** (the Greek capital letter phi) is the **cumulative probability of the standard normal "bell curve"**. You need only three facts about it:

| Phi(x) | meaning |
|---|---|
| Phi(0) = 0.5 | exactly on the line: a coin flip |
| Phi(1) = 0.84 | one s above the line: 84 percent sure it is above |
| Phi(-1) = 0.16 | one s below the line: only 16 percent chance it is above |

The ratio `(theta_u - tau) / s_u` counts **how many standard deviations the estimate is from the line**. Far above gives P_u near 1. Far below gives P_u near 0. Close to the line gives P_u near 0.5.

* **P(call right)** = P_u if the call is "above", and 1 - P_u if the call is "below".
* **P(call wrong)** = `p_wrong = 1 - P(call right)`, and it equals `min(P_u, 1 - P_u)`.

**95 percent range (card):** theta_u plus or minus **1.96 * s_u**, clipped to [0, 1]. (1.96 is the value that leaves 2.5 percent in each tail of the bell curve.)

Try it with numbers you can check by hand.

In [ ]:
import math, json, hashlib, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
TAU, PRIOR_PI, M = 0.25, np.array([0.08, 0.75]), 10.0

def Phi(x):                                   # standard normal cumulative probability, same as intake._phi
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))

print("Phi(0)=%.3f  Phi(1)=%.3f  Phi(-1)=%.3f  Phi(1.96)=%.3f  Phi(-1.96)=%.3f" % (Phi(0), Phi(1), Phi(-1), Phi(1.96), Phi(-1.96)))

def stratified_call(W, k, n, prior=PRIOR_PI, m=M, tau=TAU):
    """Same maths as intake.stratified_call."""
    W, k, n, prior = (np.asarray(a, float) for a in (W, k, n, prior))
    p = (k + m * prior) / (n + m)                         # shrunk rate in each stratum
    theta = float((W * p).sum())                          # the unit's flooded share
    var = float((W ** 2 * p * (1 - p) / (n + m + 1)).sum())
    s = math.sqrt(max(var, 1e-12))
    P = Phi((theta - tau) / s)
    return dict(p=p, theta=theta, s=s, call="above" if theta > tau else "below", P_u=P)

# A unit: 80 percent of the paddy is 'map dry', 20 percent 'map flooded'.
# Field checks so far: map dry stratum: 6 checks, 1 flooded.  map flooded stratum: 4 checks, 3 flooded.
r = stratified_call(W=[0.8, 0.2], k=[1, 3], n=[6, 4])
print("\nshrunk p per stratum:", r["p"], "  (by hand: (1+0.8)/16 = 0.1125 and (3+7.5)/14 = 0.75)")
print("theta = 0.8*0.1125 + 0.2*0.75 = %.4f" % r["theta"], "| s = %.4f" % r["s"], "| call:", r["call"], "| P_u = %.3f" % r["P_u"])
p_right = r["P_u"] if r["call"] == "above" else 1 - r["P_u"]
print("probability the call is right: %.3f   (theta is just below 0.25, so this is a near coin flip)" % p_right)
print("95 percent range: %.3f to %.3f" % (max(0, r["theta"] - 1.96 * r["s"]), min(1, r["theta"] + 1.96 * r["s"])))

In [ ]:
# Behaviour you should be able to describe: what happens as we do more checks (same true rates, more data)
rows = []
for n_each in [0, 2, 5, 10, 25, 60]:
    k = np.array([0.1 * n_each, 0.7 * n_each])            # data that look like p0=0.10, p1=0.70
    n = np.array([n_each, n_each], float)
    r = stratified_call([0.8, 0.2], k, n)
    rows.append((n_each * 2, r["theta"], r["s"], r["P_u"]))
print(pd.DataFrame(rows, columns=["total checks", "theta", "s (uncertainty)", "P_u"]).round(3).to_string(index=False))
print("\nWith 0 checks theta is just the prior blend and s is largest. More checks: theta moves to the data, s shrinks, P_u moves away from 0.5.")

# The bell curve picture: where theta sits relative to the line
x = np.linspace(-0.1, 0.7, 400)
fig, ax = plt.subplots(figsize=(6, 2.6))
th, s = 0.24, 0.0653
ax.plot(x, np.exp(-0.5 * ((x - th) / s) ** 2) / (s * np.sqrt(2 * np.pi)), label="belief about the true share")
ax.axvline(TAU, c="r", ls="--", label="loss line 0.25"); ax.axvline(th, c="k", lw=1, label="theta = 0.24")
ax.fill_between(x, 0, np.exp(-0.5 * ((x - th) / s) ** 2) / (s * np.sqrt(2 * np.pi)), where=x > TAU, alpha=.3, color="C1", label="area right of the line = P_u")
ax.legend(fontsize=7); ax.set_xlabel("true flooded share"); ax.set_yticks([]); plt.tight_layout(); plt.show()

**Say it to the guide (section 1):** "For each unit, theta is the area-weighted flooded share over the map strata. Each stratum's rate is shrunk toward a calibrated prior by a pseudo-count m of 10, so a unit with few checks is not fooled. s measures the uncertainty. The call is simply theta above or below 0.25. P_u is the normal-curve probability that the true share is above the line, Phi of theta minus tau over s. It is the probability the call is right when the call is above, and one minus it when the call is below."

## 2. What goes IN to the tool: inputs and provenance (section 1 of the notebook)

### 2.1 The contract is files

Layer 5 never imports earlier layers' code. It reads **files** from `notebooks/artifacts/layer1/` to `layer4/`. The function is `I.load_inputs(ART)` and it returns an object `Inputs` with these parts:

| Part | Columns / content | Meaning |
|---|---|---|
| `units` | unit_id, lon, lat, theta, s, call, P_u, n_checks, optional W0, W1, k0, n0, k1, n1 | the unit estimates from Layer 3 |
| `roads_nodes`, `roads_edges` | node, lon, lat / a, b, minutes, flooded | the road graph with a flooded flag per edge |
| `routes` | officer, nodes, stops, minutes | the day's routes |
| `scenes` | id, date, sensor | the satellite scenes used |
| `checks` | unit_id, stratum, flooded, pi, officer, ts | completed field checks |
| `region` | name, bbox, event | where and what |
| `provenance` | part to "real: path" or "fixture" | **what is real and what is stand-in** |

### 2.2 The fixture: a documented stand-in for anything missing

For every part whose real file does not exist (or lacks the expected columns), the tool uses a **fixture** (`make_fixture(seed=0)`), so the notebook always runs and a demo is always possible. What the fixture contains, exactly (its docstring is the specification):

* **12 invented units** on a 4 x 3 grid of 0.05-degree cells (the coordinates are only a drawing frame, not a real place);
* two map strata per unit, prior pi_h = (0.08, 0.75), m = 10;
* 0 to 8 field checks per unit with known inclusion probabilities;
* a **9 x 7 road lattice** where edges lying over a smooth flood field are flooded (**8 times slower**, `FLOOD_X = 8`);
* two officers' day routes from a small planner;
* scene IDs that start with **`SYNTH_`**, so they can never be mistaken for real Sentinel product IDs.

### 2.3 Provenance: the most important honesty feature of Layer 5

The first thing the real notebook does is print a **provenance table**: for each part, "real" or "fixture", and what each upstream layer says about its own input ("layer3 says: ..."). A file can be real but hold synthetic content, so the tool also reads each layer's `MANIFEST.json` and reports what the layer says its inputs were.

**What is actually real in our committed run (from `layer5/MANIFEST.json`):**

| Part | Source in our run |
|---|---|
| units | **real**: Layer 3's `unit_calls.csv` (**34 units**) |
| scenes | **real**: from Layer 1's build receipt |
| roads | **fixture lattice** laid over the unit layout |
| routes | **fixture planner** (Layer 4's `routes_day1` visits other units than the ones Layer 3 shows, so it is not used) |
| checks | **none** (no Layer 4 checks table) |
| unit layout | **schematic grid** (the units are scattered chips, not one region, so a real map would mislead; real lon/lat are kept in `lon_real`, `lat_real`) |
| Layer 4 says | flood layer is **semi-synthetic** |

Be ready to state this plainly: **the unit numbers and scene IDs are real; the road view and routes are stand-ins; the field checks and farmer reports typed in the demo are demonstration entries.**

In [ ]:
# Provenance in miniature: each part says where it came from, and a synthetic id is visibly synthetic
prov = {"units": "real: artifacts/layer3/unit_calls.csv", "scenes": "real: artifacts/layer1/receipts/layer1_build.json",
        "roads": "fixture lattice", "routes": "fixture planner", "checks": "none (no layer-4 checks table)"}
print(pd.DataFrame({"part": list(prov), "source": list(prov.values())}).to_string(index=False))

scene_ids = ["S1A_IW_GRDH_1SDV_20220723T004503_xxxx", "SYNTH_S1A_IW_GRDH_20260710_WET"]
for s in scene_ids:
    print(f"{s:45s}", "-> synthetic stand-in" if s.startswith("SYNTH_") else "-> looks like a real Sentinel product id")

**Say it to the guide (section 2):** "The tool reads only files from the earlier layers. Where a file is missing it uses a documented fixture with SYNTH_ scene ids so nothing synthetic can pass as real. The first output is a provenance table showing, part by part, what is real. In our committed run the 34 unit estimates and the scenes are real, the roads and routes are fixture, and there are no layer 4 field checks."

## 3. Step 1: the event board (section 2 of the notebook)

Just a display. It shows: the **region name**, the **event**, the **number of units**, the **loss line (25 percent)**, and a table of the **satellite scenes** (id, date, sensor). If Layer 1's `region.json` exists it adds "Layer 1 region: Godavari delta block (Amalapuram, Konaseema, Andhra Pradesh), season 2022-06-01 to 2022-10-31". In the old prototype this step was a free-text question box. Now the officer **starts from the event**.

## 4. Step 2: the unit-calls table and map (section 3)

### 4.1 The table (`V.unit_calls_table`)

Columns: **rank, unit_id, name, theta, s, call, P_u, p_call_right, n_checks, p_wrong.**

**Ordering rule (very important):** rows are sorted by **p_wrong descending**, so the **top row is the unit whose call is most likely wrong**. The top rows are where one more field check helps most. This is the reason the table is not sorted by flood size.

### 4.2 The map (`V.plot_unit_map`)

**Colour = the call** (above or below). **Shade = the probability the call is right** (darker = more sure). The most-likely-wrong unit is **outlined**. A Leaflet web map (via **folium**, an optional library) is also produced as `unit_map.html`; the code refuses to draw it when units are only on a schematic layout, because a real-looking web map of a fake layout would mislead.

### 4.3 The loss-line slider (`V.at_tau`)

An officer can **move the loss line** (say from 25 to 30 percent). Then P_u is recomputed as `Phi((theta - tau_new) / s)` using `scipy.stats.norm.cdf`, and the call is recomputed as `theta > tau_new`. Layer 3 numbers theta and s do not change, only the line moves.

### 4.4 The summary line

It prints counts of calls above and below, and **expected wrong calls = sum of p_wrong over all units**. That sum is an honest estimate of how many of the calls on the page are wrong. (It is an average over units of the probability of error, so it can be fractional.)

Try it on a small made-up table.

In [ ]:
units = pd.DataFrame({"unit_id": ["A", "B", "C", "D", "E"],
                      "theta":   [0.33, 0.17, 0.46, 0.05, 0.31],
                      "s":       [0.22, 0.12, 0.27, 0.14, 0.03],
                      "n_checks":[0,    0,    0,    0,    8]})

def at_tau(u, tau):
    t = u.copy(); t["P_u"] = [Phi((th - tau) / max(s, 1e-9)) for th, s in zip(u.theta, u.s)]; return t

def unit_calls_table(u, tau=0.25):
    t = at_tau(u, tau)
    t["call"] = np.where(t.theta > tau, "above", "below")
    t["p_call_right"] = np.where(t.call == "above", t.P_u, 1 - t.P_u)
    t["p_wrong"] = 1 - t.p_call_right
    t = t.sort_values("p_wrong", ascending=False).reset_index(drop=True)
    t.insert(0, "rank", np.arange(1, len(t) + 1))
    return t

t = unit_calls_table(units)
print(t.round(3).to_string(index=False))
print("\nexpected wrong calls on this page:", round(float(t.p_wrong.sum()), 2))
print("Look at E: theta=0.31 is above the line, but s=0.03 is small (8 checks), so it is well supported and NOT a doubtful row.")
print("Look at A: no checks, large s: the call is shaky, so it is rank 1.\n")

t30 = unit_calls_table(units, tau=0.30)
print("Same units with the loss line moved to 0.30:")
print(t30[["rank", "unit_id", "call", "P_u", "p_wrong"]].round(3).to_string(index=False))

**Say it to the guide (section 4):** "Every unit has a call, and the table is ordered by the chance the call is wrong, so the top rows show where a field check helps most. On the map, colour is the call and shade is the probability it is right. The officer can move the loss line and P_u is recomputed from theta and s with the normal curve. The page also reports the expected number of wrong calls, which is the sum of the per-unit wrong probabilities."

## 5. Step 3: the evidence card and the receipt (section 4)

### 5.1 What a card is

For one unit, one screen that states: **the call, theta with its 95 percent range, P_u, the probability the call is right, the number of field checks, the scenes (id and date), and a receipt**. Built by `C.build_card(unit, scenes, checks, tau, params, sources)` in `card.py`.

The call is **derived from theta, never withheld**. The builder raises an error if P_u is outside [0, 1] or theta is NaN, so a broken unit cannot silently produce a card.

### 5.2 The answer sentence

Example from our real run (unit `India_747992_u01`): "flooded paddy share 41.0% (95% range 9.7% to 72.3%). Call: above the 25% loss line. Probability the call is right: 0.84." Read the wide range: with only 3 checks the unit is still uncertain, and the card says so honestly.

### 5.3 The receipt in the card (same idea as Layer 1, new form)

A card's receipt contains:

* `inputs`: unit_id, theta, s, P_u (rounded), n_checks, **check_hashes** (sorted fingerprints of the field checks used), **scene_ids**, tau, params (m and pi_h), method version (`paddybird-tool/0.1`), sources;
* `output`: call, theta, P_u, p_call_right, range95;
* `output_hash` = SHA-256 of `{inputs, output}` in canonical JSON;
* `receipt_id` = **first 16 characters** of the SHA-256 of the whole receipt.

`verify_receipt` **re-derives both hashes** from the stored inputs and outputs and compares. If someone edits one number, verification fails.

**Reproducibility:** rebuilding a card from the same inputs gives the **same receipt id**. That is one of the six final checks.

### 5.4 Why check hashes are included

Each field check has an **entry hash** (its fingerprint). The card lists them, so the receipt proves **exactly which checks** produced this call. Add or remove a check and the receipt id changes.

### 5.5 Caveats printed on every card

* "Share of paddy area flooded is a proxy: insured-plot records are not public."
* "P_u assumes the stated variance; spatial clustering inside a unit can widen it."

**Try it: build a mini card and tamper with it.**

In [ ]:
def canonical(o):  return json.dumps(o, sort_keys=True, separators=(",", ":"), default=str)
def digest(o):     return hashlib.sha256(canonical(o).encode()).hexdigest()

def build_card(unit, scenes, check_hashes, tau=0.25):
    theta, s, P = unit["theta"], unit["s"], unit["P_u"]
    call = "above" if theta > tau else "below"                 # derived, never withheld
    p_right = P if call == "above" else 1 - P
    lo, hi = max(0.0, theta - 1.96 * s), min(1.0, theta + 1.96 * s)
    inputs = {"unit_id": unit["unit_id"], "theta": round(theta, 6), "s": round(s, 6), "P_u": round(P, 6),
              "check_hashes": sorted(check_hashes), "scene_ids": sorted(scenes), "tau": tau, "method": "paddybird-tool/0.1"}
    output = {"call": call, "theta": round(theta, 4), "P_u": round(P, 4), "p_call_right": round(p_right, 4),
              "range95": [round(lo, 4), round(hi, 4)]}
    rec = {"schema": "paddybird.receipt/1", "inputs": inputs, "output": output, "output_hash": digest({"inputs": inputs, "output": output})}
    rec["receipt_id"] = digest(rec)[:16]
    return {"unit_id": unit["unit_id"], "call": call, "theta": theta, "range95": [lo, hi], "p_call_right": p_right, "receipt": rec}

def verify_receipt(rec):
    body = {k: rec[k] for k in ("schema", "inputs", "output")}
    if digest({"inputs": body["inputs"], "output": body["output"]}) != rec["output_hash"]:
        return False
    body["output_hash"] = rec["output_hash"]
    return digest(body)[:16] == rec["receipt_id"]

u = {"unit_id": "A", "theta": 0.41, "s": 0.16, "P_u": Phi((0.41 - 0.25) / 0.16)}
card = build_card(u, ["SCENE_1", "SCENE_2"], ["a1b2", "c3d4", "e5f6"])
print("call:", card["call"], "| p(call right): %.3f" % card["p_call_right"], "| 95 percent range: %.3f to %.3f" % tuple(card["range95"]))
print("receipt id:", card["receipt"]["receipt_id"], "| verifies:", verify_receipt(card["receipt"]))
print("same inputs again, same id:", build_card(u, ["SCENE_1", "SCENE_2"], ["e5f6", "a1b2", "c3d4"])["receipt"]["receipt_id"] == card["receipt"]["receipt_id"], "(order of hashes does not matter: they are sorted)")

bad = json.loads(canonical(card["receipt"])); bad["output"]["call"] = "below"           # someone edits the answer
print("after editing the call, verifies:", verify_receipt(bad))
print("one more field check gives a different id:", build_card(u, ["SCENE_1", "SCENE_2"], ["a1b2", "c3d4", "e5f6", "zz99"])["receipt"]["receipt_id"] != card["receipt"]["receipt_id"])

**Say it to the guide (section 5):** "The evidence card gives the call, the flooded share with a 95 percent range of theta plus or minus 1.96 s, P_u, the probability the call is right, the number of checks, the scenes, and a receipt. The receipt is canonical JSON with a SHA-256 hash of the inputs and outputs, including the hashes of the field checks and the scene ids, and a 16-character receipt id. Verification recomputes both hashes, and rebuilding from the same inputs gives the same id."

## 6. Step 4: the officer day-route view (section 5)

### 6.1 What it shows

The day's **routes**, one **colour per officer** (four colours defined), over the **road graph**, with **flooded roads in blue**. Village dots are shaded by the **chance their call is wrong** (the value of a visit). It prints each officer's stops and minutes, and how many road segments are flooded.

### 6.2 Where routes come from

* When Layer 4 has written `routes_day1.json` **and** it visits the same units the table shows, the tool uses Layer 4's real routes and roads.
* Otherwise it uses a small **fallback planner** (`plan_routes`) **only so the view has something to draw**. Layer 4 owns real routing. In our committed run the fallback was used.

### 6.3 The fallback planner in plain words (cheapest insertion by gain per extra minute)

* **Value of visiting unit v:** `g(v) = min(P_u, 1 - P_u)` = the probability its call is wrong. A visit removes uncertainty, so a more doubtful unit is worth more.
* Travel times come from **Dijkstra on the flood-aware graph** (see notebook 1), where a **flooded road edge costs 8 times its dry minutes** (`FLOOD_X = 8`).
* Settings: **2 officers**, **day length 240 minutes**, **18 minutes per check** (`check_minutes`).
* Loop: for every unit not yet routed, every officer, and every position in that officer's route, compute the **extra minutes** `dt = D(a, v) + D(v, b) - D(a, b) + check_minutes` of inserting v between neighbours a and b. Keep the insertion with the best **gain per extra minute `g(v)/dt`** that still fits inside the day. Insert it. Repeat until nothing fits.
* Each route starts and ends at the **depot**.

Words: a **heuristic** is a method that finds a good answer quickly but does not guarantee the best one. Cheapest insertion is a heuristic.

Try a tiny version.

In [ ]:
rng = np.random.default_rng(0)
# 5 villages and a depot on a plane; flooded travel is slow. D = travel minutes between all points.
pts = np.array([[0, 0], [2, 1], [4, 0], [3, 3], [0, 3], [5, 2]], float)       # index 0 = depot
D = np.hypot(*(pts[:, None, :] - pts[None, :, :]).transpose(2, 0, 1)) * 5.0   # 5 min per unit distance (dry)
D[1, 2] = D[2, 1] = D[1, 2] * 8.0                                             # one road is flooded: 8 times slower
P_u = np.array([None, 0.5, 0.9, 0.2, 0.6, 0.05])
g = np.array([0] + [min(p, 1 - p) for p in P_u[1:]])                          # value of a visit = prob call is wrong
print("value of visiting villages 1..5:", np.round(g[1:], 2))

day, check, route, t = 120.0, 18.0, [], 0.0           # one officer here, a 120 minute day
left = set(range(1, 6))
while left:
    best = None
    for v in left:
        for pos in range(len(route) + 1):
            a = 0 if pos == 0 else route[pos - 1]
            b = 0 if pos == len(route) else route[pos]
            dt_ = D[a, v] + D[v, b] - D[a, b] + check          # extra minutes of inserting v between a and b
            if t + dt_ <= day and (best is None or g[v] / dt_ > best[0]):
                best = (g[v] / dt_, v, pos, dt_)
    if best is None:
        break
    _, v, pos, dt_ = best
    route.insert(pos, v); t += dt_; left.discard(v)
    print(f"insert village {v} at position {pos}: extra {dt_:.0f} min, route now 0 > {' > '.join(map(str, route))} > 0, total {t:.0f} min")
print("not visited (no time or low value):", sorted(left))

**Say it to the guide (section 6):** "The route view draws each officer's day over the road graph with flooded roads in blue and villages shaded by the chance their call is wrong. Layer 4 owns real routing. If its routes do not match the units on screen, the tool uses a cheapest-insertion fallback that adds the village with the largest probability of a wrong call per extra minute, on a graph where flooded roads cost eight times more, for two officers over a 240 minute day with 18 minutes per check."

## 7. Step 5: offline field-check entry and live update (section 6)

### 7.1 The problem

After a flood there may be **no mobile network**. Field staff must still record checks on a phone and have them merge later without double counting.

### 7.2 What one field check contains (`FieldCheck`)

`unit_id, plot_id, flooded (True/False), officer, stratum (0 map dry, 1 map flooded), pi (the known inclusion probability of the plot), optional lon, lat, photo, ts (timestamp)`. The method `record()` adds an **entry_hash** = a 16-character fingerprint of the entry **excluding the timestamp**, so the same plot, answer and officer always produce the same hash.

### 7.3 The offline store (`OfflineCheckStore`)

* An **append-only JSONL file** on the device. **JSONL** = "JSON lines": one JSON record per line. Append-only means you only ever add lines at the end, never rewrite old ones, which is robust if the phone dies mid-write.
* `add()` first checks that **0 < pi <= 1** (an invalid inclusion probability is rejected), then appends a line.
* `pending()` reads all queued lines. `clear()` empties the queue.
* `sync(checks)` merges the queue into the central table and **drops duplicates by entry_hash** (keeps the first). So if a flaky phone sends the same entry twice, it counts once. The notebook demonstrates this: **4 entries queued, one is a duplicate, exactly 3 new checks result.**

### 7.4 Updating the unit after sync (`apply_checks`)

Each new check is **weighted by 1 / pi**, normalised so that **pi = 0.5 gives weight 1**: `w = 0.5 / pi` (with pi clipped to at least 0.01). So a plot that had a low chance of being chosen (small pi) counts for more. This is the same Horvitz-Thompson idea from notebook 1.

Two paths:

1. **Strata-aware (if the unit has W0, W1, k0, n0, k1, n1):** add `w * y` to the stratum's k and `w` to its n for the check's stratum, then recompute `stratified_call` (section 1 above). Updates k, n, theta, s, call, P_u.
2. **Beta update (otherwise; this is what happens for the real Layer 3 table, which has only theta and s):** treat the unit's current belief as a **Beta distribution** with the same mean and variance, then add the new weighted checks.
   * pseudo-count `m = theta (1 - theta) / s^2 - 1` (never less than 1): "how many imaginary checks is the current belief worth";
   * `a = theta * m + k_new`, `b = (1 - theta) * m + (n_new - k_new)` where `k_new = sum(w * y)` and `n_new = sum(w)`;
   * new `theta = a / (a + b)`, new `s = sqrt(a b / ((a + b)^2 (a + b + 1)))`, then the call and `P_u = Phi((theta - tau) / s)` as before.

**Beta distribution** in one line: a distribution over a probability between 0 and 1, described by two positive numbers a and b. Think "a flooded checks, b dry checks seen so far". `a / (a + b)` is its mean. More total checks (a + b) means smaller spread.

Finally `n_checks` goes up by the number of new checks.

Here the **three demonstration entries** in the real notebook are *typed for the demo*, not field data: (flooded, stratum 1, pi 0.4), (flooded, stratum 1, pi 0.4), (dry, stratum 0, pi 0.25). The notebook shows the focus unit **before and after**: theta, s, call, P_u, n_checks.

Try it, using the same three entries and the real before-values of the top-ranked unit in our committed table (theta = 0.32625, s = 0.21789).

In [ ]:
import hashlib
def short_hash(o, n=16): return hashlib.sha256(json.dumps(o, sort_keys=True, separators=(",", ":"), default=str).encode()).hexdigest()[:n]

def beta_update(theta, s, k, n, tau=0.25):
    theta = min(max(float(theta), 1e-4), 1 - 1e-4)
    m = max(theta * (1 - theta) / max(float(s) ** 2, 1e-8) - 1.0, 1.0)      # pseudo-count the current belief is worth
    a, b = theta * m + k, (1 - theta) * m + (n - k)
    t = a / (a + b)
    sd = math.sqrt(a * b / ((a + b) ** 2 * (a + b + 1)))
    return t, sd, ("above" if t > tau else "below"), Phi((t - tau) / sd), m

# --- the offline queue, simulated in memory
queue = []
def add(chk):
    assert 0 < chk["pi"] <= 1, "inclusion probability must be in (0, 1]"
    rec = dict(chk); rec["entry_hash"] = short_hash({k: v for k, v in chk.items() if k != "ts"}); queue.append(rec); return rec

demo = [dict(unit_id="X", plot_id="X-P1", flooded=True,  officer="officer 1", stratum=1, pi=0.40, ts="t1"),
        dict(unit_id="X", plot_id="X-P2", flooded=True,  officer="officer 1", stratum=1, pi=0.40, ts="t2"),
        dict(unit_id="X", plot_id="X-P3", flooded=False, officer="officer 1", stratum=0, pi=0.25, ts="t3")]
for c in demo: add(c)
add(demo[0])                                                   # same entry queued twice by a flaky phone
print("queued:", len(queue), "entries")
try:
    add(dict(unit_id="X", plot_id="bad", flooded=True, officer="o", stratum=1, pi=1.7, ts="t"))
except AssertionError as e:
    print("rejected:", e)

# --- sync: drop duplicates by entry_hash, keep first
seen, synced = set(), []
for r in queue:
    if r["entry_hash"] not in seen: seen.add(r["entry_hash"]); synced.append(r)
print("after sync:", len(synced), "unique checks (the duplicate was dropped)")

# --- apply: each check weighted w = 0.5 / pi, then Beta update
g = pd.DataFrame(synced)
w = 0.5 / g.pi.clip(lower=0.01); y = g.flooded.astype(float)
print("weights 0.5/pi:", w.round(3).tolist(), "(small pi counts for more)")
k_new, n_new = float((w * y).sum()), float(w.sum())
before = (0.32625, 0.21789)
th, s, call, P, m = beta_update(*before, k_new, n_new)
print(f"\nbefore: theta={before[0]:.3f} s={before[1]:.3f} call={'above' if before[0] > .25 else 'below'} P_u={Phi((before[0]-.25)/before[1]):.3f}")
print(f"after : theta={th:.3f} s={s:.3f} call={call} P_u={P:.3f}   (belief was worth m={m:.1f} imaginary checks; 3 new checks, weighted total n={n_new:.2f})")
print("Notice s drops a lot: new information shrinks uncertainty, and theta moves toward what the checks found.")

**Say it to the guide (section 7):** "Field staff record checks on the phone with no network. Each check goes into an append-only JSONL queue with its inclusion probability and an entry hash. On sync, duplicates are dropped by hash, so a queue sent twice counts once. Each check is weighted by one over its inclusion probability, normalised so pi of 0.5 is weight 1. If the unit has strata we recompute the stratified estimate, otherwise we do a Beta update of theta and s. The call and P_u update immediately, and the demo entries are typed examples, not field data."

## 8. Step 6: farmer reports and the claim cross-check (section 7)

### 8.1 The situation

A farmer files an **intimation** (a loss report). Under PMFBY this must be done within **72 hours** of the loss. The tool compares the claim with the radar evidence and tells an officer **which claims to inspect first**.

**It never rejects a claim.** It only orders inspections. Self-selected reports (farmers choose to report, so they are not a random sample) **do not enter theta_u**. Say this clearly: using self-selected reports as data would be selection bias (notebook 1, section 5).

### 8.2 A farmer report (`FarmerReport`) and its record

Fields: unit_id, **farmer_ref** (stored only as a **hash**, 12 characters, for privacy), plot_ha (plot area in hectares), loss_date, reported_at, claim (default "flooded"), optional lon, lat, and `map_stratum` (0 mapped dry, 1 mapped flooded at the plot, from Layer 2). `record()` adds:

* `hours_after_loss` = reported_at minus loss_date, in hours;
* `within_72h` = True if 0 <= hours <= 72;
* `report_hash` = a fingerprint of the whole record.

### 8.3 How the probability that the plot is flooded (p) is chosen

`cross_check` picks the best available evidence, **in this order of preference:**

1. **the Layer 2 map probability at the plot** (`pixel_p`), if given;
2. else **the checked rate in the plot's map stratum**: `(k_h + m * prior_h) / (n_h + m)` (the shrunk rate from section 1) if the unit has those columns;
3. else **the calibrated prior of the plot's stratum** (`PRIOR_PI[h]`);
4. else **the unit's flooded share theta_u** (the plot has no location).

### 8.4 From p to a lean and a priority

* `consistent_prob` = p if the claim is "flooded", else 1 - p. (How likely the claim is to be true given radar.)
* **lean** = "consistent with radar" if consistent_prob >= 0.5, else "inconsistent with radar". **Always one of the two, never undecided.**
* A plot smaller than **0.5 ha** (`min_plot_ha`) is **below the radar's resolution** (10 m pixels cannot resolve it on their own). It still gets a lean, but marked **low weight** and its priority is halved.
* **Priority** = `(1 - consistent_prob) * (0.5 if small else 1.0) + (0.0 if within_72h else 0.1)`.
  So: the less the radar supports the claim, the higher the priority; small plots count half; a late report adds 0.1.
* The list is sorted by priority, highest first.

**Important reading of "inconsistent with radar":** it means *inspect this first*, not *the farmer is lying*. Radar can miss flooding under tall crops, in small plots and between passes (notebook 1, section 6).

Try it with four reports like the notebook's.

In [ ]:
PRIOR = [0.08, 0.75]
def cross_check(rep, unit, pixel_p=None, min_plot_ha=0.5):
    h = rep.get("map_stratum")
    if pixel_p is not None:           p, basis = pixel_p, "map probability at the plot"
    elif h is not None and "k0" in unit:
        p, basis = (unit[f"k{h}"] + M * PRIOR[h]) / (unit[f"n{h}"] + M), "checked rate in the plot's stratum"
    elif h is not None:               p, basis = PRIOR[h], "calibrated prior of the plot's stratum"
    else:                             p, basis = unit["theta"], "unit theta (no plot location)"
    consistent = p if rep.get("claim", "flooded") == "flooded" else 1 - p
    lean = "consistent with radar" if consistent >= 0.5 else "inconsistent with radar"
    small = rep["plot_ha"] < min_plot_ha
    priority = (1 - consistent) * (0.5 if small else 1.0) + (0.0 if rep["within_72h"] else 0.1)
    return dict(farmer=rep["farmer"], p_flooded=round(p, 3), basis=basis, consistent=round(consistent, 3), lean=lean,
                plot_ha=rep["plot_ha"], weight="low" if small else "normal", within_72h=rep["within_72h"], priority=round(priority, 3))

def hours(loss_date, reported_at):
    return (dt.datetime.fromisoformat(reported_at) - dt.datetime.fromisoformat(loss_date)).total_seconds() / 3600

unit = {"theta": 0.41, "k0": 0.5, "n0": 3.0, "k1": 4.0, "n1": 5.0}
raw = [dict(farmer="A", plot_ha=1.2, loss="2026-07-20", at="2026-07-21T10:00:00", map_stratum=1),   # map says flooded
       dict(farmer="B", plot_ha=0.8, loss="2026-07-20", at="2026-07-22T09:00:00", map_stratum=0),   # map says dry, claims flooded
       dict(farmer="C", plot_ha=0.3, loss="2026-07-20", at="2026-07-25T09:00:00", map_stratum=0),   # tiny plot, late
       dict(farmer="D", plot_ha=2.0, loss="2026-07-20", at="2026-07-21T15:00:00")]                  # no map stratum
reps = []
for r in raw:
    hrs = hours(r["loss"], r["at"]); r = dict(r, hours=round(hrs, 1), within_72h=bool(0 <= hrs <= 72)); reps.append(r)
xc = pd.DataFrame([cross_check(r, unit) for r in reps]).sort_values("priority", ascending=False)
print(xc.to_string(index=False))
print("\nB tops the list: the radar-based rate for 'map dry' is low, so the claim looks inconsistent: inspect first.")
print("C is tiny and late: low weight (half) but +0.1 for lateness. Nothing here rejects any claim.")
print("farmer id is stored hashed:", short_hash({"farmer": "farmer-A"}, 12))

**Say it to the guide (section 8):** "A farmer report is stored with the farmer reference hashed, the plot area, and the hours since the loss, because PMFBY asks for 72 hours. The cross-check estimates the probability the plot is flooded from, in order, the Layer 2 map probability at the plot, the checked rate in its map stratum, the stratum prior, or the unit's theta. It gives every report a lean, consistent or inconsistent with radar, and a priority that falls with how well radar supports the claim, halves for plots under half a hectare and adds 0.1 if late. It orders inspections. It never rejects a claim, and self-selected reports do not enter theta."

## 9. Step 7: the wizard, and the artifacts (sections 8 and 9)

### 9.1 The wizard (`V.interactive_wizard`, `V.static_wizard`)

Three steps shown to the officer: **1 Event, 2 Unit calls, 3 Evidence card**. The **interactive** version uses **ipywidgets** (sliders and drop-downs: choose the step, move the loss line, pick a unit) and only works in a live notebook. Under `nbconvert` (the command-line notebook runner) widgets do not render, so a **static fallback** prints the same three steps as plain HTML. That way the notebook always runs end to end.

### 9.2 Files written to `artifacts/layer5/`

| File | What |
|---|---|
| `unit_calls.parquet`, `unit_calls.csv` | the ranked table after the demo sync |
| `card_<unit>.json`, `card_<unit>.html` | the evidence card as data and as a page |
| `unit_map.png`, `unit_map.html` | the unit map |
| `route_view.png` | the route picture |
| `field_checks_synced.csv` | the merged field checks |
| `cross_check.json` | the farmer-report cross-check list |
| `receipts.json` | one receipt per unit |
| `MANIFEST.json` | the index with the schema, shapes, seed, provenance and hashes of the inputs and code |

The Layer 5 manifest records: **hashes of every input file** from Layers 1 to 4, **hashes of its own code** (`intake.py`, `card.py`, `views.py`), the fixture description if any part was fixture, and a note: "field checks and farmer reports in this run are typed demonstration entries, not field data". Shapes in our run: **34 units, 3 checks, 2 routes, 4 reports**.

## 10. The final checks (section 10): the six stated bounds

The checks are written **in advance**, then asserted. If any fails the notebook stops.

| # | Bound | Why it matters |
|---|---|---|
| 1 | **0 units without a call.** Every call is above or below, and the word "abstain" appears nowhere in any card | the "no abstain" rule |
| 2 | **Every P_u is in [0, 1]** and not NaN; **every receipt re-derives (100 percent)** | probabilities are valid; records are tamper-evident |
| 3 | **Rebuilding a card from the same inputs gives the same receipt id** | reproducibility |
| 4 | **The duplicate queue entry is not double counted: exactly 3 new checks**, and the focus unit's `n_checks` rose by exactly 3 | offline sync is safe |
| 5 | **Every cross-check has one of the two leans** | no "undecided" |
| 6 | **The roads table has a flooded flag on every road**; on a fixture lattice at least one flooded road is drawn in blue; and the **manifest exists** | the route view is meaningful |

Final line printed: how many units, how many without a call (0), how many receipts verified, and **which parts were real**.

**Say it to the guide (sections 9 and 10):** "The tool writes a unit-calls table, evidence cards, the map, the route view, the merged checks, the cross-check list, receipts and a manifest with hashes of the inputs and the code. Because it fits nothing it has no accuracy figure. Its checks are structural: zero units without a call, every P_u in range, every receipt re-derives, same inputs give the same receipt, duplicates are not double counted, every report has a lean, and the manifest exists."

## 11. The numbers to remember (mug this table)

| Thing | Value |
|---|---|
| Loss line **tau** | **0.25** |
| Calls | only **above** or **below** (no abstain) |
| Map strata | 2: **0 = map dry**, **1 = map flooded** |
| Prior flooded rate pi_h | **(0.08, 0.75)** |
| Shrinkage m | **10** |
| theta_u | sum over strata of W_uh * p_uh |
| p_uh | (k + m * pi_h) / (n + m) |
| P_u | Phi((theta_u - tau) / s_u) (same in Layer 3 and Layer 5) |
| P(call right) | P_u if above, 1 - P_u if below |
| p_wrong | min(P_u, 1 - P_u) |
| 95 percent range | theta plus or minus 1.96 s, clipped to [0, 1] |
| Table order | by p_wrong, largest first |
| Phi(0), Phi(1), Phi(-1) | 0.5, 0.84, 0.16 |
| Field-check weight | w = 0.5 / pi (so pi = 0.5 gives 1) |
| Beta update | m = theta(1-theta)/s^2 - 1; a = theta m + k; b = (1-theta) m + (n-k) |
| Receipt id | first **16** hex chars of a SHA-256 |
| Farmer id | hashed, **12** chars |
| Report window | **72 hours** |
| Small plot | under **0.5 ha**: low weight, priority halved |
| Priority | (1 - consistent) * (0.5 if small else 1) + (0 if within 72 h else 0.1) |
| Flooded road cost | **8 times** the dry minutes (`FLOOD_X`) |
| Fallback planner | **2** officers, **240 min** day, **18 min** per check, cheapest insertion by gain per extra minute |
| Fixture | **12** invented units, **9 x 7** lattice, ids start with **SYNTH_** |
| Our run | **34** units (real, from Layer 3), **3** demo checks, **2** routes, **4** reports |
| Real in our run | unit estimates, scenes. **Fixture**: roads, routes. **None**: layer 4 checks |
| Demo entries | typed for the notebook, **not field data** |
| Version string | `paddybird-tool/0.1` |

## 12. Glossary (alphabetical)

* **Above / below**: the two possible calls: estimated flooded share is above or at/below tau.
* **Abstain**: refuse to answer. Forbidden in this project.
* **Append-only**: a file that is only added to, never rewritten.
* **Beta distribution**: a distribution for a probability, described by a and b (about "a floods and b dry seen").
* **Call**: the decision for a unit, above or below the loss line.
* **Card (evidence card)**: the one-screen record of a unit's call, numbers, scenes, checks and receipt.
* **Entry hash**: fingerprint of one field check (excluding the timestamp), used to drop duplicates.
* **Fixture**: a documented stand-in dataset used when the real file is missing.
* **Folium**: a Python library that draws Leaflet web maps.
* **Heuristic**: a quick method that gives a good but not guaranteed-best answer.
* **Hajek / Horvitz-Thompson weight**: weight 1 over the inclusion probability, to undo unequal chances of selection.
* **Inclusion probability (pi)**: the known chance a plot was selected. Not the same as the prior pi_h.
* **Intimation**: the farmer's loss report.
* **JSONL**: one JSON record per line.
* **Lean**: whether a claim is consistent or inconsistent with radar. Always one of the two.
* **m**: shrinkage strength (10), the prior's worth in imaginary checks.
* **n_checks**: the number of field checks counted for a unit.
* **P_u**: probability the unit's true flooded share is above tau.
* **p_wrong**: probability the call is wrong, min(P_u, 1 - P_u).
* **Phi**: the standard normal cumulative probability.
* **PMFBY**: the crop-insurance scheme; 72-hour intimation rule.
* **Prior (pi_h)**: the belief about a stratum's flooded rate before looking at this unit.
* **Provenance**: where each part of the input came from (real file or fixture).
* **Receipt / receipt id**: canonical JSON with hashes, and its 16-character id.
* **s_u**: the uncertainty (standard error) of theta_u.
* **Schematic layout**: units drawn on a plain grid because they have no common map region.
* **Shrinkage**: pulling an estimate from few data points toward a prior.
* **Stratum / strata**: a group of the unit's area (map dry, map flooded).
* **Sync**: merging the phone's queued checks into the central table.
* **tau**: the loss line, 0.25.
* **theta_u**: the estimated share of the unit's paddy area that is flooded.
* **W_uh**: the share of the unit's area in stratum h.
* **Wizard**: the three-step interface: Event, Unit calls, Evidence card.

## 13. Questions an examiner may ask, and the one-line answers

1. **What does Layer 5 do?** It presents Layers 1 to 4 to the officer: unit calls, map, evidence card, route view, offline entry, claim cross-check.
2. **What is its accuracy?** It has none. It fits nothing. Its checks are structural.
3. **Why no abstain state?** Every unit needs a decision. The uncertainty is carried by P_u instead of withholding an answer.
4. **What is theta_u?** The estimated share of a unit's paddy area that is flooded: sum over map strata of area share times flooded rate.
5. **What is P_u?** The probability the true share is above 25 percent: Phi of (theta minus tau) over s.
6. **What is Phi?** The cumulative probability of the standard normal curve. Phi(0) is 0.5.
7. **How do you get the probability the call is right?** P_u if the call is above, one minus P_u if below.
8. **Why is the table sorted by chance of being wrong?** Doubtful units are where an extra check helps most.
9. **What is shrinkage and why?** Pulling a small-sample rate toward a prior so two checks cannot fool the estimate. Strength m = 10.
10. **What is the evidence card?** The call, flooded share with a 95 percent range, P_u, check count, scenes, and a receipt.
11. **What is in a receipt and how is it verified?** Canonical JSON of inputs and outputs, with SHA-256 hashes. Verification re-derives them.
12. **How does the tool work offline?** Checks are queued in an append-only JSONL file with an entry hash, and merged on sync with duplicates dropped.
13. **How is a field check weighted?** By one over its inclusion probability, normalised so pi of 0.5 is 1. It undoes unequal selection chances.
14. **What is the Beta update?** Treat the unit's current theta and s as a Beta belief, add the weighted new checks, and read off the new theta and s.
15. **What does the farmer cross-check do?** Gives each report a lean and an inspection priority. It never rejects a claim and does not feed theta.
16. **Why not use farmer reports to estimate theta?** They are self-selected, so the sample is biased.
17. **Why is a plot under 0.5 ha low weight?** The 10 m radar cannot resolve it on its own.
18. **Where do routes come from?** Layer 4. If its routes do not match the units shown, a fallback cheapest-insertion planner draws something.
19. **What is real and what is not in your run?** Real: 34 unit estimates and the scenes. Fixture: roads and routes. No layer 4 checks. Demo field checks and reports are typed examples.
20. **What happens if a part is missing?** A documented fixture takes its place, scene ids start with SYNTH_, and provenance says so.
21. **Why does the wizard have a static fallback?** Widgets do not render under nbconvert, so a static HTML version keeps the notebook runnable.
22. **What are the final checks?** No unit without a call, P_u in range and receipts re-derive, same inputs same receipt, no double counting, every report has a lean, flooded flag and manifest present.